<a href="https://colab.research.google.com/github/manuelavasquezo/Trabajos-IA-/blob/main/2_Taller_Apriori_Rodriguez_Mendoza_Vasquez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Miguel Andrés Rodriguez
- Dana Melissa Mendoza
- Manuela Vásquez Osorio

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [ ]:
### TU CÓDIGO AQUÍ ###
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [ ]:
### TU CÓDIGO AQUÍ ###
df =  pd.read_csv('/content/drive/MyDrive/Copia de alimentos_retail_top30.csv')

In [ ]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [ ]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [ ]:
### TU CÓDIGO AQUÍ ###
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,20
Country,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [ ]:
### TU CÓDIGO AQUÍ ###
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

En la columna Quantity se observa que el valor mínimo es negativo, porque una cantidad vendida en un contexto normal de ventas no puede ser negativa.
Podría asumirse que estos valores probablemente representen devoluciones o cancelaciones, sin embargo, lo ideal sería tratarlos por separado antes del análisis, ya que el algoritmo apriori solo debe trabajar con transacciones de compras reales.

En cuanto a la columna Customer ID, se observa que tiene un count distinto al resto de columnas. Teniendo esta 6.879 y las demás 6.899, lo que significa que hay 20 registros sin identificador de cliente. Esto es un problema porque no se puede saber a qué cliente pertenecen esas transacciones, dificultando la agrupación de compras por cliente para construir las canastas de mercado.

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [ ]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
df_limpio = df.dropna(subset=['CustomerID'])

In [ ]:
# El tipo de dato de CustomerID debe ser entero
### TU CÓDIGO AQUÍ ###
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [ ]:
# TAREA: # Verifica cuántas descripciones únicas hay.
### TU CÓDIGO AQUÍ ###
df_limpio['Description'].nunique()

25

In [ ]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
### TU CÓDIGO AQUÍ ###
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [ ]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
### TU CÓDIGO AQUÍ ###
df_limpio['Description'].nunique()


20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [ ]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
### TU CÓDIGO AQUÍ ###
df_limpio = df_limpio[~df_limpio['InvoiceNo'].astype(str).str.startswith('C')]

In [ ]:
# TAREA: Elimina las filas con cantidades negativas.
### TU CÓDIGO AQUÍ ###
df_limpio = df_limpio[df_limpio['Quantity'] > 0]

In [ ]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [ ]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df_limpio, 'México')

In [ ]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_mx = apriori(
    cesta_mx,
    min_support=0.02,
    use_colnames=True
)
frequent_itemsets_mx.sort_values(by='support', ascending=False).head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(CHILE JALAPEÑO, TOMATE)"


In [ ]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
### TU CÓDIGO AQUÍ ###
rules_mx = association_rules(
    frequent_itemsets_mx,
    metric='lift',
    min_threshold=2
)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
rules_mx.sort_values(['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]

,antecedents,consequents,antecedent support,consequent support,confidence,lift
91,"(CEBOLLA, CHILE JALAPEÑO)","(CILANTRO, TOMATE)",0.32,0.32,0.92,2.90
90,"(CILANTRO, TOMATE)","(CEBOLLA, CHILE JALAPEÑO)",0.32,0.32,0.93,2.90
88,"(CILANTRO, CEBOLLA)","(CHILE JALAPEÑO, TOMATE)",0.31,0.33,0.95,2.88
93,"(CHILE JALAPEÑO, TOMATE)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
11,"(AGUACATE, LIMÓN)",(TOTOPOS),0.27,0.33,0.93,2.82
14,(TOTOPOS),"(AGUACATE, LIMÓN)",0.33,0.27,0.75,2.82
89,"(CILANTRO, CHILE JALAPEÑO)","(CEBOLLA, TOMATE)",0.32,0.33,0.92,2.81
92,"(CEBOLLA, TOMATE)","(CILANTRO, CHILE JALAPEÑO)",0.33,0.32,0.91,2.81
73,"(AGUACATE, TOMATE, LIMÓN)",(TOTOPOS),0.02,0.33,0.91,2.76
76,(TOTOPOS),"(AGUACATE, TOMATE, LIMÓN)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Las reglas 1 y 3 giran en torno a 4 ingredientes, cebolla, chile jalapeño, cilantro y tomate. Ingredientes que componen la base del pico de gallo o picadillo. Esto indica un patrón de compra cultural claro en el cual los clientes mexicanos adquieren estos ingredientes juntos de forma consistente como parte de una preparación tradicional.

La regla 5 muestra una asociación entre aguacate, limón con un consecuente de totopos, lo que corresponde al consumo de guacamole y que también constituye una salsa común en México.

Son ingredientes frescos de cocina tradicional mexicana, todos de la categoría de perecederos/frescos (verduras, frutas y snacks). No son productos procesados ni de marca, sino insumos para preparaciones caseras culturalmente arraigadas.

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1: el soporte de antecedentes es de 0.32 lo que significa que el 32% de las transacciones en México incluyes los ingredientes de cebolla y chille jalapeño. El soporte del consecuente también indica que el tomate y cilantro aparecen juntos en un 32% de las transacciones. La confianza es del 92% indicando la probabilidad que hay de que las veces que un cliente compra cebolla y chile, también compra cilantro y tomate en la misma transacción. Finalmente, el lift es de 2.90 y confirma que está asociación no es casualidad, comprar cilantro y tomate es 2.90 veces más probable cuando el cliente ya lleva cebolla y chille jalapeño, en comparación con lo que ocurriría si los productos fueran independientes entre sí.

Regla 3: el soporte del antecedente es 0.31, es decir, el 31% de las transacciones contienen cilantro y cebolla juntos. El soporte consecuente es 0.33, indicando que el chile jalapeño y tomate aparecen juntos en el 33% de las compras. Esta regla tiene una confianza de 0.95, lo que significa que el 95% de los clientes que compran cilantro y cebolla también se llevan chile jalapeño y tomate. El loft de 2.88 refuerza que la relación es fuerte y no es aleatoria, cuando un cliente compra cilantro y cebolla, tiene una probabilidad de 2.88 veces más de comprar chile jalapeño y tomate.

Regla 5: el soporte de antecedente indica que el 27% de las transacciones en México incluyen aguacate y limón juntos. El soporte consecuente es 0.33, indicando que los totopos aparecen solos en el 33% de las compras. La confianza de 0.93 muestra que el 93% de las veces que un cliente lleva aguacate y limón, también agrega totopos a su compra. El lift de 2.82 confirma que la asociación es significativa pues comprar totopos es 2.82 veces más probable cuando el cliente ya tiene aguacate y limón, en comparación con la probabilidad base de comprar totopos de forma independiente.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Las reglas 1 y 3 muestran que cebolla, chile jalapeño, cilantro y tomate se compran juntos con una confianza superior al 92%, por lo que crear un paquete promocional con estos cuatro ingredientes, ofreciendo un descuento del 10% al 15%, incrementaría el ticket promedio y facilitaría la experiencia de compra del cliente mexicano.
La regla 5 revela que quien compra aguacate y limón también lleva totopos en el 93% de los casos. Esto justifica agrupar estos tres productos en un exhibidor conjunto en tienda o en una sección destacada del sitio web, reforzado con una promoción del tipo "completa tu guacamole con descuento en totopos".



3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co =  preparar_cesta(df_limpio, 'Colombia')

In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co = apriori(
    cesta_co,
    min_support=0.02,
    use_colnames=True
)
frequent_itemsets_co.sort_values(by='support', ascending=False).head(10)

,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(CAFÉ, LECHE)"
31,0.32,"(AZÚCAR, LECHE)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co = association_rules(
    frequent_itemsets_co,
    metric='lift',
    min_threshold=2
)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
rules_co.sort_values(['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]

,antecedents,consequents,antecedent support,consequent support,confidence,lift
40,"(CAFÉ, FRIJOL CARGAMANTO, AZÚCAR)",(LECHE),0.05,0.39,0.96,2.44
45,(LECHE),"(CAFÉ, FRIJOL CARGAMANTO, AZÚCAR)",0.39,0.05,0.11,2.44
10,"(CAFÉ, AZÚCAR)",(LECHE),0.32,0.39,0.95,2.41
15,(LECHE),"(CAFÉ, AZÚCAR)",0.39,0.32,0.76,2.41
5,"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",(ARROZ),0.30,0.38,0.91,2.41
8,(ARROZ),"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",0.38,0.30,0.73,2.41
58,"(CAFÉ, PAN TAJADO, AZÚCAR)",(LECHE),0.03,0.39,0.94,2.39
67,(LECHE),"(CAFÉ, PAN TAJADO, AZÚCAR)",0.39,0.03,0.07,2.39
48,"(HUEVOS, CAFÉ, AZÚCAR)",(LECHE),0.04,0.39,0.92,2.36
57,(LECHE),"(HUEVOS, CAFÉ, AZÚCAR)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

Las reglas 1 y 3 muestran un patrón claro alrededor del café y azúcar con la leche, que es una preparación cotidiana en Colombia. La relación de por qué el frijol cargamanto se encuentra en la regla 1 no es muy clara, sin embargo podría sugerir que los colombianos realizan comprar integrales para su canasta, en la que incluyen compras para cualquier alimento que se pueda preparar durante el día.
En la regla 5, podríamos observar y reforzar la idea de la canasta básica. Pues el aceite de girasol, frijol y arroz son ingredientes básicos que se usan en gran parte de las preparaciones colombianas.


Estos patrones son claramente diferentes a los de México, donde las asociaciones giraban en torno a ingredientes frescos para preparaciones muy específicas como el pico de gallo o guacamole. En Colombia, las reglas reflejan más una lógica de abastecimiento de despensa con productos no perecederos y de consumo diario.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1: El soporte de antecedentes indica que solo el 5% de las transacciones en Colombia contienen estos tres productos juntos, siendo una combinación menos frecuente pero de igual forma especifica. El soporte consecuente es de 0.39, es decir, la leche aparece en el 39% de todas las transacciones, lo que la convierte en un producto altamente comprado por los colombianos. La confianza indica que el 96% de las veces que alguien compra café, frijol y azúcar juntos, también lleva leche. Finalmente, el lift de 2.44 confirma que esta asociación es significativa pues comprar leche es 2.44 veces más probable cuando el cliente lleva esos tres productos.

Regla 3: el soporte del antecedente es 0.32, lo que significa que casi un tercio de todas las transacciones en Colombia incluyen café y azúcar juntos. El soporte consecuentes es 0.39, al igual que en la regla anterior, confirma el peso de la leche en la canasta colombiana. La confianza de 0.95 indica que el 95% de quienes compran café y azúcar también llevan leche. El log de 2.41 muestra que esta asociación es casi 2.5 veces más fuerte que si los productos fueran independientes.

Regla 5: el soporte de antecedentes es 0.30, indicando que el 30% de las transacciones incluyen aceite de girasol y frijol cargamanto juntos, una frecuencia bastante alta que habla de su rol como productos esenciales de la canasta.  El soporte del consecuente es 0.38, indicando que el arroz está presente en el 38% de todas las compras, siendo también un producto ancla de la canasta básica. La confianza muestra que el 91% de quienes compran aceite y frijol también llevan arroz. El lift de 2.41 confirma que esta relación no es aleatoria y tiene un valor predictivo real para el negocio.



3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

Las reglas 1 y 3 muestran que café, azúcar y leche forman una triada inseparable en el consumidor colombiano, con confianzas del 95% y 96%. Se recomienda crear un bundle promocional con estos tres productos, posicionado como el "kit del desayuno perfecto", ofreciendo un descuento por la compra conjunta. Esta estrategia tiene un alcance masivo dado que el café y la leche tienen soportes del 32% y 39% respectivamente, lo que significa que impacta a una porción enorme de la base de clientes.

La regla 5 revela que arroz, frijol cargamanto y aceite de girasol se compran juntos de forma muy predecible. Se recomienda agrupar estos productos en un exhibidor o sección digital bajo el concepto de "todo para tu almuerzo", complementado con sugerencias de otros ingredientes como carne o verduras. Esta estrategia apunta a posicionar la plataforma como el destino preferido para el abastecimiento semanal del hogar colombiano.

Mientras que en México las promociones deben apelar a recetas concretas y culturalmente simbólicas, en Colombia la estrategia más efectiva es la de abastecimiento inteligente de despensa, resaltando el ahorro y la conveniencia de completar la canasta básica en un solo lugar.